# <center>Veille Technique

In [3]:
import numpy as np
import pandas as pd

In [ ]:
import numpy as np
import gensim.downloader as api
from mteb import MTEB
import logging

# 1. Définition du modèle "Wrapper" pour GloVe
class GloveMTEBModel:
    def __init__(self, model_name="glove-wiki-gigaword-300"):
        print(f"Chargement du modèle {model_name}...")
        self.model = api.load(model_name)
        self.vector_size = self.model.vector_size

    def encode(self, sentences, **kwargs):
        """
        Transforme une liste de phrases en une matrice de vecteurs.
        Stratégie : Mean Pooling (Moyenne des vecteurs de mots).
        """
        all_embeddings = []
        for sentence in sentences:
            # Tokenisation simple par espaces (standard pour GloVe)
            tokens = sentence.lower().split()
            # On récupère les vecteurs des mots présents dans le vocabulaire
            vectors = [self.model[w] for w in tokens if w in self.model]

            if len(vectors) > 0:
                # Moyenne des vecteurs
                mean_vec = np.mean(vectors, axis=0)
            else:
                # Vecteur nul si aucun mot n'est connu (cas rare)
                mean_vec = np.zeros(self.vector_size)

            all_embeddings.append(mean_vec)
            
        return np.array(all_embeddings)

# 2. Initialisation du modèle
my_glove_model = GloveMTEBModel()

# 3. Sélection d'une tâche spécifique (pour aller vite)
# Ici 'STS12' (Similarité Sémantique) ou 'Banking77Classification'
tasks = ["STS12"]

# 4. Exécution du benchmark
evaluation = MTEB(tasks=tasks)
results = evaluation.run(my_glove_model, output_folder="results/glove")

print("Évaluation terminée. Les résultats sont dans le dossier 'results/glove'.")